# Question format for model-based ED triage on KTAS

Same patients, same model, different **question formats**:

| arm | calls / set of 5 | what the model is asked |
|---|---|---|
| `direct_set` | 1 | all 5 patients in one prompt, a KTAS level for each |
| `direct_each` | 5 | one patient per call, one digit (expected level from logprobs if the API returns them) |
| `optionwise` | 25 | "is level k correct? Yes/No" for k = 1..5 (off by default, costly) |
| `pair_bt` / `pair_ahp` | 10 | each pair once: who first + Saaty strength, aggregated by Bradley-Terry / AHP eigenvector |
| `pair_bt_swap` / `pair_ahp_swap` | 20 | each pair in both A/B orders |
| `pairprob_bt` / `pairprob_ahp` | 20 | "A or B?" single letter, P(A first) from logprobs, both orders, log-odds aggregated |
| `nurse` | 0 | the triage nurse's KTAS_RN (human reference) |

Score: Kendall tau-b between the model's priority order and the **expert KTAS** (`KTAS_expert`) within each set; plus top-1 correct, under/over-triage (level arms), order-flip rate, calls per set and **tau gain per extra call**.

**Honesty rules.** Only report numbers this notebook prints. The `mock` backend is a fake model for testing the code: never report its numbers. If an API stops (quota), the run stops cleanly; rerun the same cell later and it resumes from the cache.

Runtime: Gemini free tier about 15 requests/min, so 10 sets x 46 calls is about 30 min per model (cached pilot calls are free).

## 1. Get the code
Either set `REPO_URL` to your GitHub copy, or leave it empty and upload `ktas-pairwise-triage.zip` when asked.

In [ ]:
import os, subprocess, sys
REPO_URL = ""   # e.g. "https://github.com/<you>/ktas-pairwise-triage.git"
if not os.path.exists("ktriage"):
    if REPO_URL:
        subprocess.run(["git", "clone", "-q", REPO_URL, "repo"], check=True)
        os.chdir("repo")
    elif os.path.exists("/content"):
        from google.colab import files
        up = files.upload()                       # choose ktas-pairwise-triage.zip
        z = next(iter(up))
        subprocess.run(["unzip", "-q", "-o", z], check=True)
        os.chdir("ktas-pairwise-triage")
print("working dir:", os.getcwd())
!pip -q install -r requirements.txt

## 2. API keys
In Colab: left sidebar, key icon (Secrets), add `GEMINI_API_KEY` (from aistudio.google.com) and, for any OpenAI-compatible provider, `OPENAI_API_KEY`. Keys are read from the environment and never written to disk.

In [ ]:
try:
    from google.colab import userdata
    for k in ("GEMINI_API_KEY", "OPENAI_API_KEY"):
        try:
            v = userdata.get(k)
            if v:
                os.environ[k] = v
        except Exception:
            pass
except ImportError:
    pass
print({k: ("set" if os.environ.get(k) else "missing") for k in ("GEMINI_API_KEY", "OPENAI_API_KEY")})

## 3. Offline smoke test (no API, about 5 seconds)
Checks the whole pipeline with the fake `mock` model. Its numbers mean nothing.

In [ ]:
!python -m pytest -q tests

## 4. Choose models

* Gemini: `gemini-3.5-flash-lite` reuses the 477 cached pilot calls (10 sets), so only the new arms cost calls. Check the exact model names your key can use at aistudio.google.com and add more.
* Decision models (Gemini Decision 1, Liquid d1-ultra, Jev, ...): add them **only** if the provider gives you an OpenAI-compatible endpoint, a model id and a key. Fill `base_url` and `model` from the provider's docs. Nothing here has been tested against those APIs.
* Local open model: run section 4b first (needs a GPU runtime), then keep the `ollama` entry.

In [ ]:
N_SETS = 10          # sets of 5 patients (the pilot used 10 of these 20)
ARMS = ["direct_set", "direct_each", "pair_bt", "pair_ahp", "pair_bt_swap", "pair_ahp_swap",
        "pairprob_bt", "pairprob_ahp"]          # add "optionwise" if you have quota (25 calls/set)

MODELS = [
    {"model": "gemini-3.5-flash-lite", "backend": "gemini", "rpm": 14, "max_calls": 480},
    # {"model": "<second gemini model>", "backend": "gemini", "rpm": 9, "max_calls": 480},
    # {"model": "qwen2.5:7b", "backend": "openai", "base_url": "http://localhost:11434/v1", "rpm": 0, "max_calls": 5000},
    # {"model": "<decision model id>", "backend": "openai", "base_url": "<provider OpenAI-compatible URL>", "rpm": 30, "max_calls": 1000},
]

### 4b. (Optional) local open model with Ollama in Colab
Runtime > Change runtime type > T4 GPU. No quota; about 10-20 min for 10 sets with a 7B model.

In [ ]:
USE_OLLAMA = False
if USE_OLLAMA:
    !curl -fsSL https://ollama.com/install.sh | sh
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    import time; time.sleep(5)
    !ollama pull qwen2.5:7b
    MODELS.append({"model": "qwen2.5:7b", "backend": "openai", "base_url": "http://localhost:11434/v1",
                   "rpm": 0, "max_calls": 5000})

### 4c. Gemini Decisions API (`gemini-decision-1`), the decision-model arm
Uses the same `GEMINI_API_KEY`. First run the probe: it sends ONE request and prints the raw answer. If it prints `HTTP 404` or `403`, your key cannot use the Decisions API yet; skip 4c. The request format comes from third-party guides (12 Nov 2026), so check the printed JSON has `answers` with `probabilities` / `noul`.

In [ ]:
from ktriage import decisions
raw = decisions.probe("gemini-decision-1")

In [ ]:
# 20 sets: direct_each (ordered rubric), optionwise (yes/no per level), pairprob (A/B choice, both orders), direct_set
# about 31 requests per set = ~620 requests; free tier is reportedly 200 requests/min, 10,000/day (verify in AI Studio)
RUN_DECISIONS = True
if RUN_DECISIONS and raw is not None:
    from ktriage.experiment import run
    _, stop = run("gemini-decision-1", "decision", n_sets=20,
                  arms=["direct_set", "direct_each", "optionwise", "pairprob_bt", "pairprob_ahp"],
                  seed=42, data_csv="data/ktas_data.csv", out_root="results", rpm=150, max_calls=2000, workers=4)
    print("status:", stop or "complete")

## 5. Run
Every answer is cached in `results/<model>/cache.jsonl`. If a quota stops a model, rerun this cell later; it resumes.

In [ ]:
from ktriage.experiment import run
seed_cache = ["data/pilot_cache_gemini-3.5-flash-lite.jsonl"]
status = {}
for m in MODELS:
    try:
        _, stop = run(m["model"], m["backend"], N_SETS, ARMS, seed=42, data_csv="data/ktas_data.csv",
                      out_root="results", rpm=m.get("rpm", 14), max_calls=m.get("max_calls", 480),
                      workers=4, base_url=m.get("base_url"), seed_caches=seed_cache)
        status[m["model"]] = stop or "complete"
    except Exception as e:
        status[m["model"]] = f"FAILED: {e}"
status

## 6. Results table and figure
`common_only=True`: arms are compared on the same sets only. `gain_per_call` = paired tau gain over `direct_each` divided by the extra calls.

In [ ]:
import pandas as pd
from ktriage.analysis import load_all, summarise, plot_calls_vs_tau
per_set = load_all("results")
per_set = per_set[per_set.backend != "mock"]          # never report mock numbers
summary = summarise(per_set, ref="direct_each", common_only=True)
summary.to_csv("results/summary.csv", index=False)
fig = plot_calls_vs_tau(summary, "results/calls_vs_tau.png")
pd.set_option("display.width", 250)
display(summary.round(3))
from IPython.display import Image
Image(fig)

## 7. Plain-language readout for the paper
Copy these lines; every number comes from `results/summary.csv`.

In [ ]:
for model, g in summary.groupby("model"):
    g = g.set_index("arm")
    print(f"\n{model} (n = {int(g.n_sets.max())} sets of 5)")
    for arm, r in g.iterrows():
        line = f"  {arm:14s} calls/set {int(r.calls_per_set):2d}  tau {r.tau_mean:.3f} [{r.tau_ci_lo:.3f}, {r.tau_ci_hi:.3f}]  top-1 {r.top1:.2f}"
        if pd.notna(r.get("gain_vs_ref", float('nan'))):
            line += f"  gain vs {r.ref} {r.gain_vs_ref:+.3f} [{r.gain_ci_lo:+.3f}, {r.gain_ci_hi:+.3f}], per call {r.gain_per_call:+.4f}"
        if pd.notna(r.get("under", float('nan'))):
            line += f"  undertriage {r.under:.2f}"
        if isinstance(r.get("source"), str) and r.source:
            line += f"  probs: {r.source}"
        print(line)

## 8. Download everything

In [ ]:
!zip -q -r results.zip results -x '*/cache.jsonl'
!zip -q -r results_with_cache.zip results
try:
    from google.colab import files
    files.download("results.zip")
except ImportError:
    print("results.zip written")